# Урок 2. Кросс-валидация: честная оценка модели

Идея: одно разбиение = один экзамен, результат зависит от удачи. Кросс-валидация делит данные на 5 частей
и проводит 5 экзаменов: каждый пассажир ровно один раз в тесте. Итог = среднее.

## 0. Подготовка

In [1]:
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

df = pd.read_csv("../data/titanic.csv")
df["Sex_num"] = df["Sex"].map({"male": 0, "female": 1})
y = df["Survived"]

# SimpleImputer внутри pipeline: медиана считается только по учебной части каждого раунда (нет утечки)
model = make_pipeline(SimpleImputer(strategy="median"),
                      LogisticRegression(max_iter=1000))

## 1. Одна модель, 5 экзаменов

In [2]:
cv5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, df[["Pclass", "Sex_num", "Age", "Fare"]], y, cv=cv5)
print("5 результатов:", scores.round(3))
print(f"среднее = {scores.mean():.3f}, разброс (std) = {scores.std():.3f}")

5 результатов: [0.81  0.792 0.781 0.764 0.787]
среднее = 0.787, разброс (std) = 0.015


**Вывод:** честная оценка модели 0.787 ± 0.015. Одно число из урока 1 (0.788) было близко, но могло быть и 0.76, и 0.81.

## 2. Какие признаки важны? 5 частей против 10

In [3]:
feature_sets = [
    ("Только Sex_num",         ["Sex_num"]),
    ("Sex_num + Pclass",       ["Sex_num", "Pclass"]),
    ("Sex_num + Pclass + Age", ["Sex_num", "Pclass", "Age"]),
    ("Все 4 признака",         ["Pclass", "Sex_num", "Age", "Fare"]),
]
cv10 = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

rows = []
for name, feats in feature_sets:
    for cv in [cv5, cv10]:
        scores = cross_val_score(model, df[feats], y, cv=cv)
        rows.append({"частей": cv.get_n_splits(), "признаки": name,
                     "mean": scores.mean(), "std": scores.std()})

results = pd.DataFrame(rows)
results.round(3)

,частей,признаки,mean,std
0,5,Только Sex_num,0.787,0.019
1,10,Только Sex_num,0.787,0.030
2,5,Sex_num + Pclass,0.787,0.019
3,10,Sex_num + Pclass,0.787,0.030
4,5,Sex_num + Pclass + Age,0.786,0.016
5,10,Sex_num + Pclass + Age,0.782,0.038
6,5,Все 4 признака,0.787,0.015
7,10,Все 4 признака,0.786,0.039


**Вывод 1. Почему точность почти одинаковая (≈ 0.787)?**
Модель `Sex_num + Pclass` даёт **те же ответы**, что правило «женщины выжили». Класс меняет вероятности
(по модели: мужчина 1-го класса 43%, 3-го — 10%), но все мужчины остаются ниже 50% → «погиб», все женщины выше 50% → «выжил».
Ответы те же, значит, и точность та же. Accuracy не видит вероятностей (это видит AUC — урок 4).

**Вывод 2. Почему при 10 частях std больше?**
В каждом тесте меньше людей (~89 вместо ~178), и одна ошибка сильнее меняет точность.
Среднее почти не изменилось (0.787 против 0.786), а разброс вырос с 0.015 до 0.039.